# Cancer PRO Decline Prediction: Preprocessing Pipeline

**Project Context:**
This notebook is the second stage of our ML mini project predicting a 10+ point decline in PROMIS Global Physical Health (`drop_outcome`) for cancer patients on chemotherapy (based on Ostberg & Peterson, Stanford CS229).

**Scope of this Notebook:**
This notebook focuses strictly on **data preprocessing** (no model training). We transform the raw dataset into clean, scaled, leakage-free numeric matrices ready for machine learning algorithms.

**Key Clinical & ML Principles Implemented:**
1. **Zero Data Leakage:** We strictly drop `on_chemo_gph` (which was used to compute the labels) and `threshold_outcome` (an alternative post-chemo label) before modeling.
2. **Train/Test Split First:** We partition the data into an 80% training set and a 20% test set *before* fitting any imputer, scaler, or encoder.
3. **Stratification:** We preserve the ~20.4% positive class ratio across both splits.
4. **Clinical Imputation with Missingness Indicators:** Missing laboratory values are imputed using training medians, and binary indicator columns are created to preserve the predictive signal of missingness.
5. **Standardization & One-Hot Encoding:** Numeric features are standardized to zero mean and unit variance; categorical features are one-hot encoded with unseen-category safety.
6. **Artifact Persistence:** Processed feature matrices and the fitted `ColumnTransformer` are saved for downstream model training and deployment.


# Step 1: Load Data and Define Target

### What this step does
In this step, we import the required libraries (`pandas`, `scikit-learn`, `joblib`, and `os`), load the raw synthetic dataset (`synthetic_chemo_pro_data.csv`) from `../data/raw/`, and extract our target variable `y = df["drop_outcome"]`.

### Why we do this
- **Target definition:** Our primary clinical objective is predicting `drop_outcome` (binary: 1 if physical health drops by 10 or more points during chemotherapy, 0 otherwise).
- **Early separation:** Isolating the target vector $y$ immediately clarifies the distinction between input features $X$ and the outcome to be predicted.


In [1]:
# Import required libraries
import os
import pandas as pd
import joblib

# Import scikit-learn preprocessing and model selection modules
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Load the raw synthetic cancer patient dataset
df = pd.read_csv("../data/raw/synthetic_chemo_pro_data.csv")

# Define the target variable y
y = df["drop_outcome"]

# Display basic information about the loaded dataset
print("Dataset loaded successfully.")
print(f"Total rows: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")
print(f"Target variable: drop_outcome")
print(f"Target class distribution:\n{y.value_counts()}")


Dataset loaded successfully.
Total rows: 1,800
Total columns: 56
Target variable: drop_outcome
Target class distribution:
drop_outcome
0    1433
1     367
Name: count, dtype: int64


### Why I did this:
- *Placeholder: Explain why `drop_outcome` is our primary target variable for predicting physical health decline.*
- *Placeholder: Note the number of rows and columns loaded from the raw CSV.*


# Step 2: Drop Identifier, Leakage, and Secondary Outcome Columns

### What this step does
In this step, we remove four columns from our feature matrix $X$:
1. `patient_id`
2. `on_chemo_gph`
3. `threshold_outcome`
4. `drop_outcome`

### Why we do this
- **`patient_id` (Identifier):** A synthetic patient identifier. It contains no clinical signal. Keeping it could allow models to memorize IDs rather than learning generalizable medical patterns.
- **`on_chemo_gph` (Critical Data Leakage!):** This score is measured *during* chemotherapy and was directly used to construct the target label ($on\_chemo\_gph - baseline\_gph \le -10$). In a clinical deployment setting, an oncologist wants to predict decline *before* chemotherapy begins. If `on_chemo_gph` were left in $X$, any model would achieve artificially perfect accuracy by simply calculating the difference, making the model clinically useless.
- **`threshold_outcome` (Secondary Outcome):** An alternative post-treatment clinical label (score $\le 40$). It is an outcome measurement, not an input feature available prior to treatment.
- **`drop_outcome` (Target):** The target variable itself must not remain inside the feature set $X$.


In [2]:
# List the non-predictive, leakage, and outcome columns to drop
columns_to_drop = ["patient_id", "on_chemo_gph", "threshold_outcome", "drop_outcome"]

# Create feature matrix X by dropping these columns
X = df.drop(columns=columns_to_drop)

# Verify the features remaining in X
print(f"Original column count: {df.shape[1]}")
print(f"Columns dropped: {len(columns_to_drop)} -> {columns_to_drop}")
print(f"Remaining feature count in X: {X.shape[1]}")
print(f"Feature matrix shape: {X.shape}")


Original column count: 56
Columns dropped: 4 -> ['patient_id', 'on_chemo_gph', 'threshold_outcome', 'drop_outcome']
Remaining feature count in X: 52
Feature matrix shape: (1800, 52)


### Why I did this:
- *Placeholder: Explain what data leakage means and why keeping `on_chemo_gph` in features would be fatal to the project.*
- *Placeholder: Why is `patient_id` excluded from the feature set?*


# Step 3: Train / Test Split (80/20, Stratified, Random State 42)

### What this step does
In this step, we split our feature matrix $X$ and target vector $y$ into:
- **Training set (80%):** 1,440 patient records
- **Test set (20%):** 360 patient records

We perform this split using `train_test_split` with `stratify=y` and `random_state=42`.

### Why we do this
- **Golden Rule of Machine Learning:** The train/test split **must occur BEFORE any preprocessing fitting**. Imputers, scalers, and encoders must compute their parameters (medians, means, standard deviations, category mappings) strictly on the training set. Fitting transformers on the full dataset before splitting leaks information from the test set into the model, leading to overly optimistic and invalid performance estimates.
- **Stratification (`stratify=y`):** Because only ~20.4% of patients experience a 10+ point drop, a random unstratified split could accidentally yield unequal proportions of decline in the train and test partitions. Stratification guarantees identical class balances across both sets.
- **Reproducibility (`random_state=42`):** Setting a fixed random seed ensures that anyone re-running the notebook gets the exact same train and test splits.


In [3]:
# Perform stratified 80/20 train/test split BEFORE any preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Print split shapes and patient counts
print("Train/test split completed successfully.")
print(f"X_train shape: {X_train.shape} (1,440 patients, 52 raw features)")
print(f"X_test shape:  {X_test.shape} (360 patients, 52 raw features)")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")


Train/test split completed successfully.
X_train shape: (1440, 52) (1,440 patients, 52 raw features)
X_test shape:  (360, 52) (360 patients, 52 raw features)
y_train shape: (1440,)
y_test shape:  (360,)


### Why I did this:
- *Placeholder: Why must train/test splitting happen BEFORE fitting scalers or imputers?*
- *Placeholder: Why is stratification critical when dealing with an imbalanced outcome like `drop_outcome`?*


# Step 4: Build ColumnTransformer for Numeric and Categorical Features

### What this step does
In this step, we:
1. Automatically detect numeric versus categorical feature names from the data types (`dtypes`) of `X_train`.
2. Construct a **numeric pipeline**:
   - `SimpleImputer(strategy="median", add_indicator=True)`: Replaces missing values with the median of each feature in the training set and appends binary indicator columns marking which values were originally missing.
   - `StandardScaler()`: Scales continuous features to have zero mean and unit variance.
3. Construct a **categorical transformer**:
   - `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`: Converts nominal and categorical variables into binary one-hot dummy columns.
4. Assemble both into an scikit-learn `ColumnTransformer`.

### Why we do this
- **Median Imputation:** Median is robust to extreme outliers in clinical laboratory markers (such as CRP or PSA).
- **Missingness Indicator (`add_indicator=True`):** In oncology EHR data, whether a test was ordered or omitted is itself a meaningful clinical signal. For example, `psa_ng_ml` is missing primarily in female patients and non-prostate cancers. Adding indicator flags preserves this clinical context for the models.
- **Standardization (`StandardScaler`):** Many ML models (Logistic Regression, Neural Networks, SVMs, Ridge/Lasso) are sensitive to feature scales. Standardizing prevents high-magnitude features (e.g., platelets in thousands) from dominating low-magnitude features (e.g., albumin).
- **Safe One-Hot Encoding (`handle_unknown="ignore"`):** If an unseen category appears in future test batches, the encoder sets all dummy columns to zero instead of throwing an error.


In [4]:
# Detect numeric columns (int64 and float64) from X_train
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Detect categorical columns (object dtype) from X_train
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

# Print summary of detected column types
print(f"Detected {len(numeric_cols)} numeric columns:")
print(numeric_cols)
print(f"\nDetected {len(categorical_cols)} categorical columns:")
print(categorical_cols)

# Build pipeline for numeric features: Median Imputation + Missing Indicator, then StandardScaler
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

# Build transformer for categorical features: OneHotEncoder
categorical_pipeline = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Assemble into ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ],
    remainder="drop"
)

print("\nColumnTransformer successfully constructed.")
print(preprocessor)


Detected 44 numeric columns:
['age', 'bmi', 'stage', 'ecog_status', 'prior_surgery', 'prior_radiation', 'months_since_diagnosis', 'num_comorbidities', 'diabetes', 'hypertension', 'copd', 'heart_disease', 'depression_history', 'med_analgesic', 'med_opioid', 'med_antidepressant', 'med_antiemetic', 'med_steroid', 'num_med_classes', 'ed_visits_6m', 'hospitalizations_6m', 'psychiatry_visits_6m', 'heart_rate', 'systolic_bp', 'diastolic_bp', 'temperature_c', 'resp_rate', 'spo2', 'weight_loss_pct_6m', 'hemoglobin_g_dl', 'wbc_k_ul', 'platelets_k_ul', 'albumin_g_dl', 'creatinine_mg_dl', 'alt_u_l', 'bilirubin_mg_dl', 'sodium_mmol_l', 'potassium_mmol_l', 'glucose_mg_dl', 'crp_mg_l', 'ldh_u_l', 'psa_ng_ml', 'baseline_gph', 'baseline_gmh']

Detected 8 categorical columns:
['sex', 'race', 'insurance', 'marital_status', 'smoking_status', 'cancer_type', 'chemo_regimen', 'treatment_intent']

ColumnTransformer successfully constructed.
ColumnTransformer(transformers=[('num',
                             

### Why I did this:
- *Placeholder: Why is median imputation preferred over mean imputation for medical lab data?*
- *Placeholder: Why is `add_indicator=True` clinically useful for missing lab tests?*
- *Placeholder: Why is `StandardScaler` important for distance-based and regularized algorithms?*


# Step 5: Fit on `X_train` and Transform Both Sets

### What this step does
In this step, we:
1. **Fit and transform** `X_train` using `preprocessor.fit_transform(X_train)`.
2. **Transform** `X_test` using `preprocessor.transform(X_test)`.
3. Extract the clean transformed feature names using `preprocessor.get_feature_names_out()`.
4. Wrap the resulting NumPy arrays back into pandas DataFrames (`X_train_processed` and `X_test_processed`) with column names and original indices intact.
5. Print shapes before and after transformation.

### Why we do this
- **Strict training-set fitting:** Calling `.fit()` solely on `X_train` ensures that all transformation parameters (medians, scaling means/stds, one-hot category vocabularies) are learned exclusively from the training cohort.
- **Unbiased evaluation on `X_test`:** Calling `.transform()` on `X_test` applies those exact learned parameters to test patients, properly simulating how the model will process new, unseen patient data in production.
- **Feature tracking:** Converting transformed outputs back into labeled pandas DataFrames ensures we can track individual feature names (e.g., identifying which coefficients or feature importances belong to which lab test or category).


In [5]:
# Fit the preprocessor on X_train and transform X_train
X_train_processed_array = preprocessor.fit_transform(X_train)

# Transform X_test using the preprocessor fitted on X_train only (NEVER call fit on test data!)
X_test_processed_array = preprocessor.transform(X_test)

# Retrieve transformed feature names (including imputed features, indicators, and one-hot categories)
feature_names = preprocessor.get_feature_names_out()

# Convert transformed arrays into pandas DataFrames with feature names and original patient indices
X_train_processed = pd.DataFrame(
    X_train_processed_array,
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed_array,
    columns=feature_names,
    index=X_test.index
)

# Print shapes before and after transformation
print("=== Shapes Before Preprocessing ===")
print(f"X_train (raw): {X_train.shape}")
print(f"X_test (raw):  {X_test.shape}")

print("\n=== Shapes After Preprocessing ===")
print(f"X_train_processed: {X_train_processed.shape}")
print(f"X_test_processed:  {X_test_processed.shape}")
print(f"Total features generated: {len(feature_names)}")

# Display a preview of the first 5 processed rows and first 6 columns
print("\nFirst 5 rows of X_train_processed (sample columns):")
X_train_processed.iloc[:5, :6]


=== Shapes Before Preprocessing ===
X_train (raw): (1440, 52)
X_test (raw):  (360, 52)

=== Shapes After Preprocessing ===
X_train_processed: (1440, 89)
X_test_processed:  (360, 89)
Total features generated: 89

First 5 rows of X_train_processed (sample columns):


,num__age,num__bmi,num__stage,num__ecog_status,num__prior_surgery,num__prior_radiation
957,2.299679,-1.788237,-1.799931,0.094087,0.919866,1.620865
309,1.796303,1.131913,1.201112,0.094087,-1.087115,-0.616954
630,1.628512,1.880201,-1.799931,-1.208654,0.919866,-0.616954
962,-0.972260,0.055108,1.201112,0.094087,-1.087115,-0.616954
345,-2.062906,-1.642229,0.200764,1.396828,-1.087115,1.620865


### Why I did this:
- *Placeholder: Why did we call `fit_transform` on `X_train` but only `transform` on `X_test`?*
- *Placeholder: Why did the column count increase from 52 to 89 (hint: missing indicators + one-hot encoded categories)?*


# Step 6: Data Quality Check (NaNs and Class Balance)

### What this step does
In this step, we perform two quality checks:
1. Verify that there are zero missing (`NaN`) values remaining in both `X_train_processed` and `X_test_processed`.
2. Verify the class balance of `drop_outcome` in `y_train` and `y_test`.

### Why we do this
- **Completeness verification:** Most standard machine learning algorithms in scikit-learn will raise a `ValueError` if passed missing values. Verifying 0 NaNs confirms that our imputation pipeline executed cleanly across all features.
- **Stratification verification:** Checking the percentage of positive cases (`drop_outcome = 1`) in both sets confirms that our stratified split accurately preserved the ~20.4% clinical event rate in both training (1,440 patients) and test (360 patients) cohorts.


In [6]:
# Check for any remaining missing values in X_train_processed and X_test_processed
train_nans = X_train_processed.isna().sum().sum()
test_nans = X_test_processed.isna().sum().sum()

print("=== Missing Values Quality Check ===")
print(f"Total NaNs in X_train_processed: {train_nans}")
print(f"Total NaNs in X_test_processed:  {test_nans}")

if train_nans == 0 and test_nans == 0:
    print("Check Passed: Feature matrices are clean with zero missing values.")
else:
    print("Warning: Missing values still exist!")

# Check class balance in y_train and y_test
print("\n=== Class Balance Verification ===")
train_counts = y_train.value_counts().sort_index()
train_pcts = (y_train.value_counts(normalize=True).sort_index() * 100).round(2)
train_balance_df = pd.DataFrame({"Count": train_counts, "Percentage (%)": train_pcts})

test_counts = y_test.value_counts().sort_index()
test_pcts = (y_test.value_counts(normalize=True).sort_index() * 100).round(2)
test_balance_df = pd.DataFrame({"Count": test_counts, "Percentage (%)": test_pcts})

print("Training set class balance (y_train):")
print(train_balance_df)
print("\nTest set class balance (y_test):")
print(test_balance_df)


=== Missing Values Quality Check ===
Total NaNs in X_train_processed: 0
Total NaNs in X_test_processed:  0
Check Passed: Feature matrices are clean with zero missing values.

=== Class Balance Verification ===
Training set class balance (y_train):
              Count  Percentage (%)
drop_outcome                       
0              1146           79.58
1               294           20.42

Test set class balance (y_test):
              Count  Percentage (%)
drop_outcome                       
0               287           79.72
1                73           20.28


### Why I did this:
- *Placeholder: Confirm that zero NaNs remain in both datasets.*
- *Placeholder: Confirm that the positive class proportion (~20.4%) matches between training and test sets.*


# Step 7: Save Processed Datasets and Fitted Preprocessor

### What this step does
In this final step, we persist all preprocessing artifacts:
1. Save `X_train_processed` to `../data/processed/X_train.csv` (with column names from `get_feature_names_out`).
2. Save `X_test_processed` to `../data/processed/X_test.csv`.
3. Save `y_train` to `../data/processed/y_train.csv`.
4. Save `y_test` to `../data/processed/y_test.csv`.
5. Save the fitted `preprocessor` pipeline object to `../models/preprocessor.joblib`.

### Why we do this
- **Modular workflow:** By saving the processed data, our next notebook (`03_model_training.ipynb`) can immediately load ready-to-train datasets without repeating the preprocessing computation.
- **Model deployment and inference:** Serializing `preprocessor.joblib` allows us to preprocess brand-new, unseen patient records in production or web applications (`app/`) using the exact transformation rules learned on the training cohort.


In [7]:
# Ensure output directories exist
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# Define file paths for processed data
path_x_train = "../data/processed/X_train.csv"
path_x_test = "../data/processed/X_test.csv"
path_y_train = "../data/processed/y_train.csv"
path_y_test = "../data/processed/y_test.csv"
path_preprocessor = "../models/preprocessor.joblib"

# Save feature matrices and targets to CSV
X_train_processed.to_csv(path_x_train, index=False)
X_test_processed.to_csv(path_x_test, index=False)
y_train.to_csv(path_y_train, index=False)
y_test.to_csv(path_y_test, index=False)

# Save the fitted ColumnTransformer preprocessor using joblib
joblib.dump(preprocessor, path_preprocessor)

# Verify and print saved artifact details
print("=== Processed Artifacts Successfully Saved ===")
print(f"X_train saved to: {path_x_train} (Size: {os.path.getsize(path_x_train) / 1024:.1f} KB)")
print(f"X_test  saved to: {path_x_test}  (Size: {os.path.getsize(path_x_test) / 1024:.1f} KB)")
print(f"y_train saved to: {path_y_train} (Size: {os.path.getsize(path_y_train) / 1024:.1f} KB)")
print(f"y_test  saved to: {path_y_test}  (Size: {os.path.getsize(path_y_test) / 1024:.1f} KB)")
print(f"Preprocessor saved to: {path_preprocessor} (Size: {os.path.getsize(path_preprocessor) / 1024:.1f} KB)")


=== Processed Artifacts Successfully Saved ===
X_train saved to: ../data/processed/X_train.csv (Size: 1768.1 KB)
X_test  saved to: ../data/processed/X_test.csv  (Size: 443.5 KB)
y_train saved to: ../data/processed/y_train.csv (Size: 4.2 KB)
y_test  saved to: ../data/processed/y_test.csv  (Size: 1.1 KB)
Preprocessor saved to: ../models/preprocessor.joblib (Size: 8.7 KB)


### Why I did this:
- *Placeholder: Explain the purpose of saving `X_train.csv`, `X_test.csv`, `y_train.csv`, and `y_test.csv`.*
- *Placeholder: Why is saving `preprocessor.joblib` important for downstream model deployment or new patient predictions?*


# Step 8: Label Verification (Target Integrity)

### What this step does
In this step, we programmatically verify the integrity of the binary target variable `drop_outcome`. We recompute the label directly from raw clinical PROMIS scores using the formula defined by Ostberg & Peterson:
$$\text{drop\_outcome} = \begin{cases} 1 & \text{if } (on\_chemo\_gph - baseline\_gph) \le -10 \\ 0 & \text{otherwise} \end{cases}$$
We then assert that our recomputed values match the `drop_outcome` column in the dataset with zero mismatches, and report the overall positive rate.

### Why we do this
- **Target verification:** Verifying ground truth prevents silent target misalignment before model training.
- **Clinical definition audit:** Confirming the exact mathematical relationship ensures the target definition reflects a clinically meaningful functional decline (1 standard deviation drop in physical health).


In [8]:
# Recompute drop_outcome directly from raw PROMIS scores
recomputed_drop = ((df["on_chemo_gph"] - df["baseline_gph"]) <= -10).astype(int)

# Calculate the number of mismatches between the original column and recomputed label
mismatches = (df["drop_outcome"] != recomputed_drop).sum()
positive_rate = (df["drop_outcome"].mean()) * 100

# Assert that there are zero mismatches
assert mismatches == 0, f"Label verification failed: {mismatches} mismatches detected between drop_outcome and formula!"

print("=== Step 8: Label Verification Summary ===")
print(f"Total patient records verified: {len(df):,}")
print(f"Number of mismatches:           {mismatches}")
print(f"Cohort positive rate:           {positive_rate:.2f}% ({df['drop_outcome'].sum():,} / {len(df):,})")
print("Verification Result: PASSED - drop_outcome exactly matches (on_chemo_gph - baseline_gph) <= -10.")


=== Step 8: Label Verification Summary ===
Total patient records verified: 1,800
Number of mismatches:           0
Cohort positive rate:           20.39% (367 / 1,800)
Verification Result: PASSED - drop_outcome exactly matches (on_chemo_gph - baseline_gph) <= -10.


# Step 9: Pipeline and Artifact Consistency Checks

### What this step does
In this step, we run a battery of rigorous automated assertions (`assert`) to ensure our processed data and serialized artifacts satisfy all machine learning engineering standards:
1. Confirm `X_train_processed` and `X_test_processed` contain identical feature columns in the exact same order.
2. Confirm no leaking, secondary outcome, or identifier columns (`on_chemo_gph`, `threshold_outcome`, `drop_outcome`, `patient_id`) appear in the feature sets.
3. Confirm sample lengths align perfectly between feature matrices and target vectors.
4. Confirm zero `NaN` or missing values remain in either processed matrix.
5. Confirm that the positive class proportion in `y_train` and `y_test` are within 1 percentage point of each other.
6. Reload saved CSV files and `preprocessor.joblib` from disk to confirm dimensions and feature schemas match in-memory data.

### Why we do this
Automated consistency checks act as defensive unit tests for the data preprocessing pipeline, catching silent errors (such as feature misalignment, leaked columns, or partition imbalance) before model training begins in Notebook 03.


In [9]:
# Consistency Check 1: Identical columns in exact same order
assert list(X_train_processed.columns) == list(X_test_processed.columns), (
    "Column mismatch: X_train_processed and X_test_processed do not have identical columns in the same order!"
)
print("[PASS] Consistency Check 1: X_train_processed and X_test_processed have identical columns in the same order.")

# Consistency Check 2: No leakage, outcome, or identifier columns in features
forbidden_columns = ["on_chemo_gph", "threshold_outcome", "drop_outcome", "patient_id"]
for forbidden in forbidden_columns:
    assert forbidden not in X_train_processed.columns, f"Forbidden column '{forbidden}' found in X_train_processed!"
    assert forbidden not in X_test_processed.columns, f"Forbidden column '{forbidden}' found in X_test_processed!"
print("[PASS] Consistency Check 2: No leakage or identifier columns (on_chemo_gph, threshold_outcome, drop_outcome, patient_id) in features.")

# Consistency Check 3: Feature and target sample counts match
assert len(y_train) == len(X_train_processed), (
    f"Train length mismatch: len(y_train)={len(y_train)} != len(X_train_processed)={len(X_train_processed)}"
)
assert len(y_test) == len(X_test_processed), (
    f"Test length mismatch: len(y_test)={len(y_test)} != len(X_test_processed)={len(X_test_processed)}"
)
print(f"[PASS] Consistency Check 3: Sample lengths match (Train: {len(y_train)}, Test: {len(y_test)}).")

# Consistency Check 4: Zero NaNs in processed matrices
assert X_train_processed.isna().sum().sum() == 0, "NaN values detected in X_train_processed!"
assert X_test_processed.isna().sum().sum() == 0, "NaN values detected in X_test_processed!"
print("[PASS] Consistency Check 4: Zero NaNs found in processed train and test matrices.")

# Consistency Check 5: Class balance between train and test within 1 percentage point
train_positive_rate = y_train.mean()
test_positive_rate = y_test.mean()
rate_difference = abs(train_positive_rate - test_positive_rate)
assert rate_difference < 0.01, (
    f"Class balance discrepancy exceeds 1 percentage point: Train={train_positive_rate*100:.2f}%, Test={test_positive_rate*100:.2f}%"
)
print(f"[PASS] Consistency Check 5: Train and test positive rates are within 1 percentage point "
      f"(Train: {train_positive_rate*100:.2f}%, Test: {test_positive_rate*100:.2f}%, Diff: {rate_difference*100:.3f}%).")

# Consistency Check 6: Reload saved CSVs and preprocessor.joblib from disk and verify shapes match
reloaded_X_train = pd.read_csv("../data/processed/X_train.csv")
reloaded_X_test = pd.read_csv("../data/processed/X_test.csv")
reloaded_y_train = pd.read_csv("../data/processed/y_train.csv")
reloaded_y_test = pd.read_csv("../data/processed/y_test.csv")
reloaded_preprocessor = joblib.load("../models/preprocessor.joblib")

assert reloaded_X_train.shape == X_train_processed.shape, "Reloaded X_train shape mismatch with in-memory matrix!"
assert reloaded_X_test.shape == X_test_processed.shape, "Reloaded X_test shape mismatch with in-memory matrix!"
assert len(reloaded_y_train) == len(y_train), "Reloaded y_train length mismatch!"
assert len(reloaded_y_test) == len(y_test), "Reloaded y_test length mismatch!"
assert len(reloaded_preprocessor.get_feature_names_out()) == X_train_processed.shape[1], (
    "Reloaded preprocessor feature output count mismatch!"
)
print("[PASS] Consistency Check 6: All disk artifacts reloaded successfully; shapes and schema match in-memory data exactly.")


[PASS] Consistency Check 1: X_train_processed and X_test_processed have identical columns in the same order.
[PASS] Consistency Check 2: No leakage or identifier columns (on_chemo_gph, threshold_outcome, drop_outcome, patient_id) in features.
[PASS] Consistency Check 3: Sample lengths match (Train: 1440, Test: 360).
[PASS] Consistency Check 4: Zero NaNs found in processed train and test matrices.
[PASS] Consistency Check 5: Train and test positive rates are within 1 percentage point (Train: 20.42%, Test: 20.28%, Diff: 0.139%).
[PASS] Consistency Check 6: All disk artifacts reloaded successfully; shapes and schema match in-memory data exactly.
